# 28 — Modern NLP Task Heads & Pretrained-Model Interface

**Learning objective.** Map common NLP tasks to model outputs, losses, tensor shapes and standard pretrained-model interfaces.

This notebook follows the track contract: concept → inspectable implementation → rendered result → failure modes → production implication.

## Mental model

**shared hidden states → task-specific head/loss → task logits/embeddings → prediction**

Follow the information transformation first; treat the API as an implementation detail.

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Swap **sequence head → token head** | output tensor gains sequence axis | loss/labels/evaluation all change |
| Change pooling token/strategy | which hidden state represents sequence changes | classification logits move |
| Freeze vs fine-tune encoder | number of trainable parameters changes | data requirement/cost/adaptation trade off |

> Write down what should move downstream before changing a control.

## Think before running the next cell

1. Why can't a sequence-classification head directly return entity spans?
2. What shape should token-classification logits have for batch B, sequence T, classes C?

### When to use
Use task heads to match a shared encoder to a clearly defined output contract.

### When not to use / caution
Do not choose a head based on library convenience before defining labels and metric.

### Debugging lens
Start with tensor shapes: input IDs → hidden states → head output → target shape → loss.

In [1]:
from pathlib import Path
import re, random, math, json
import numpy as np
import pandas as pd
np.random.seed(42); random.seed(42)
print("Reproducibility seed: 42")

Reproducibility seed: 42


In [2]:
tasks=pd.DataFrame([
 ['sequence classification','[batch, labels]','cross entropy','accuracy / macro-F1','AutoModelForSequenceClassification'],
 ['token classification','[batch, seq, labels]','token cross entropy','entity/span F1','AutoModelForTokenClassification'],
 ['extractive QA','start + end logits','2× cross entropy','Exact Match / token F1','AutoModelForQuestionAnswering'],
 ['causal LM','[batch, seq, vocab]','next-token CE','perplexity + task eval','AutoModelForCausalLM'],
 ['seq2seq generation','[batch, target_seq, vocab]','target-token CE','ROUGE/BLEU + human/task eval','AutoModelForSeq2SeqLM'],
 ['embedding / retrieval','[batch, dim]','contrastive/ranking','Recall@k / MRR / nDCG','SentenceTransformer / encoder'],
],columns=['task','output shape','typical loss','evaluation','common HF interface'])
tasks

                      task  ...                 common HF interface
0  sequence classification  ...  AutoModelForSequenceClassification
1     token classification  ...     AutoModelForTokenClassification
2            extractive QA  ...       AutoModelForQuestionAnswering
3                causal LM  ...                AutoModelForCausalLM
4       seq2seq generation  ...               AutoModelForSeq2SeqLM
5    embedding / retrieval  ...       SentenceTransformer / encoder

[6 rows x 5 columns]

In [3]:
import torch, torch.nn as nn
B,T,H,C,V=2,5,16,3,50
hidden=torch.randn(B,T,H)
seq_head=nn.Linear(H,C)(hidden[:,0])
token_head=nn.Linear(H,C)(hidden)
lm_head=nn.Linear(H,V)(hidden)
print('sequence classification logits:',tuple(seq_head.shape))
print('token classification logits   :',tuple(token_head.shape))
print('language-model logits         :',tuple(lm_head.shape))

sequence classification logits: (2, 3)
token classification logits   : (2, 5, 3)
language-model logits         : (2, 5, 50)


### Standard pretrained workflow
1. Choose a checkpoint matched to language/domain/license/size.
2. Load its tokenizer and model with compatible classes.
3. Build a task-specific dataset and collator.
4. Fine-tune or use frozen embeddings/prompting depending on the task.
5. Evaluate against a baseline on a held-out split and relevant slices.
6. Export/version **tokenizer + model + label mapping + config** together.

The optional `requirements-optional-transformers.txt` lists Hugging Face packages without making the core offline track depend on remote checkpoint downloads.

---
## Production takeaways
- Treat decoding, privacy, robustness and task heads as explicit system design choices.
- Keep evaluation aligned with the actual task and deployment risk.